### Import Dataset From Drive

- Test: https://drive.google.com/file/d/1ALveidCmKzk4p_liQ7b663BXL0PcTaFN/view?usp=drive_link
- Train: https://drive.google.com/file/d/1Ujh37a1kTarNf3dSOSrj-hKDswULebYA/view?usp=drive_link

In [1]:
import pandas as pd

In [2]:
!gdown --id '1ALveidCmKzk4p_liQ7b663BXL0PcTaFN'
test = pd.read_csv('test_transaction.csv')

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1ALveidCmKzk4p_liQ7b663BXL0PcTaFN
From (redirected): https://drive.google.com/uc?id=1ALveidCmKzk4p_liQ7b663BXL0PcTaFN&confirm=t&uuid=1f3db15f-d09a-4b5e-a832-463f9214ca26
To: /content/test_transaction.csv
100% 613M/613M [00:06<00:00, 94.5MB/s]


In [3]:
!gdown --id '1Ujh37a1kTarNf3dSOSrj-hKDswULebYA'
train = pd.read_csv('train_transaction.csv')

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1Ujh37a1kTarNf3dSOSrj-hKDswULebYA
From (redirected): https://drive.google.com/uc?id=1Ujh37a1kTarNf3dSOSrj-hKDswULebYA&confirm=t&uuid=e45d51c2-a6a3-489c-af6d-087c38ee02a0
To: /content/train_transaction.csv
100% 683M/683M [00:13<00:00, 50.5MB/s]


### Check dataset

In [4]:
test.shape

(506691, 393)

In [5]:
train.shape

(590540, 394)

### Check IsFraud

In [6]:
train['isFraud'].head(5)

,isFraud
0,0
1,0
2,0
3,0
4,0


### Import Lib

LightGBM dipilih karena paling efisien untuk tabular data besar
dengan RAM terbatas (Colab free = ~12GB RAM)

In [7]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import gc
import warnings
warnings.filterwarnings('ignore')

Dataset memiliki 590k+ rows dan 394 kolom. </br>
Pandas default pakai float64/int64 yang boros RAM. </br>
Fungsi ini downcast ke tipe data terkecil yang masih muat

In [8]:
# Reduce memory saat loading
def reduce_mem_usage(df):
  for col in df.columns:
    col_type = df[col].dtype
    if col_type != object:
      c_min = df[col].min()
      c_max = df[col].max()
      if str(col_type)[:3] == 'int':
        if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
          df[col] = df[col].astype(np.int8)
        elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
          df[col] = df[col].astype(np.int16)
        elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
          df[col] = df[col].astype(np.int32)
      else:
        if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
          df[col] = df[col].astype(np.float16)
        elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
          df[col] = df[col].astype(np.float32)
  return df

### EDA

Fokus pada 3 hal penting:
  1. Distribusi target (class imbalance)
  2. Persentase missing value per kolom
  3. Tipe data kolom

In [9]:
# Class imbalance check
fraud_ratio = train['isFraud'].value_counts(normalize=True)
print(fraud_ratio)

isFraud
0    0.96501
1    0.03499
Name: proportion, dtype: float64


In [10]:
# Missing values overview
missing = train.isnull().sum()
missing_pct = (missing / len(train) * 100).sort_values(ascending=False)
print(missing_pct[missing_pct > 0].head(20))

dist2    93.628374
D7       93.409930
D13      89.509263
D14      89.469469
D12      89.041047
D6       87.606767
D9       87.312290
D8       87.312290
V153     86.123717
V149     86.123717
V141     86.123717
V146     86.123717
V154     86.123717
V162     86.123717
V142     86.123717
V158     86.123717
V161     86.123717
V157     86.123717
V138     86.123717
V139     86.123717
dtype: float64


In [ ]:
# check tipe data
print(train.dtypes.value_counts())

In [ ]:
# check statistik TransactionAmt
print(train['TransactionAmt'].describe())

### Preprocessing

Langkah:
  1. Pisah fitur (X) dan target (y)
  2. Missing value:
  - Numerik  -> fill -999 (LightGBM bisa handle nilai sentinel ini)
  - Kategori -> fill 'unknown'
  3. Label Encoding untuk kolom kategori (string -> integer), Fit pada gabungan train+test supaya tidak ada unseen label saat prediksi

In [11]:
# Pisah target
y = train['isFraud'].copy()
X = train.drop(['isFraud', 'TransactionID'], axis=1)
X_test = test.drop(['TransactionID'], axis=1)

In [12]:
# Simpan TransactionID untuk submission
test_ids = test['TransactionID']

del train, test
gc.collect()

60

In [13]:
# check MISSING VALUES

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object']).columns.tolist()

X[num_cols] = X[num_cols].fillna(-999)
X_test[num_cols] = X_test[num_cols].fillna(-999)

X[cat_cols] = X[cat_cols].fillna('unknown')
X_test[cat_cols] = X_test[cat_cols].fillna('unknown')

In [14]:
# === LABEL ENCODING untuk categorical ===
for col in cat_cols:
  le = LabelEncoder()
  # Fit on combined untuk avoid unseen labels
  combined = pd.concat([X[col], X_test[col]], axis=0)
  le.fit(combined)
  X[col] = le.transform(X[col])
  X_test[col] = le.transform(X_test[col])

### Feature Engineering

Menambahkan fitur baru yang relevan untuk fraud detection:

  1. Aggregasi TransactionAmt per grup (card, email domain)
  </br>Fraud sering terjadi pada kartu/email tertentu dengan pola amount yang berbeda dari rata-ratanya
  2. Waktu transaksi (jam & hari dari TransactionDT)
  </br>Fraud cenderung terjadi di jam-jam tertentu (misal tengah malam)
  3. Log transform TransactionAmt
  </br>Distribusi amount sangat skewed, log1p membantu model

In [15]:
# TransactionAmt statistics per card/email
for col in ['card1', 'card4', 'card6', 'P_emaildomain']:
  if col in X.columns:
    # Mean & std TransactionAmt per group
    group_mean = X.groupby(col)['TransactionAmt'].transform('mean')
    group_std = X.groupby(col)['TransactionAmt'].transform('std')
    X[f'{col}_amt_mean'] = group_mean
    X[f'{col}_amt_std'] = group_std

    # Sama untuk test (pakai mapping dari train)
    amt_mean_map = X.groupby(col)['TransactionAmt'].mean()
    amt_std_map = X.groupby(col)['TransactionAmt'].std()
    X_test[f'{col}_amt_mean'] = X_test[col].map(amt_mean_map)
    X_test[f'{col}_amt_std'] = X_test[col].map(amt_std_map)

In [16]:
# TransactionDT -> hour of day, day of week (cyclic patterns fraud)
if 'TransactionDT' in X.columns:
  X['hour'] = (X['TransactionDT'] / 3600) % 24
  X['day'] = (X['TransactionDT'] / (3600 * 24)) % 7
  X_test['hour'] = (X_test['TransactionDT'] / 3600) % 24
  X_test['day'] = (X_test['TransactionDT'] / (3600 * 24)) % 7

In [17]:
# Transaction amount log transform (skewed distribution)
X['TransactionAmt_log'] = np.log1p(X['TransactionAmt'])
X_test['TransactionAmt_log'] = np.log1p(X_test['TransactionAmt'])

gc.collect()
print(f"Final shape - X: {X.shape}, X_test: {X_test.shape}")

Final shape - X: (590540, 403), X_test: (506691, 403)


### Handling imbalance

Imbalance handling:
  - scale_pos_weight = jumlah negatif / jumlah positif
  - Lebih efisien dari SMOTE (tidak perlu generate data sintetis)

Model: LightGBM
  - Histogram-based: jauh lebih cepat & hemat RAM vs XGBoost
  - max_bin=63 (default 255): kurangi memori histogram

Validasi: StratifiedKFold 5-fold
  - Menjaga proporsi fraud di setiap fold
  - OOF (Out-of-Fold) prediction untuk evaluasi tidak bias

Early stopping:
  - Hentikan training jika AUC tidak membaik setelah 50 round
  - Mencegah overfitting dan hemat waktu

In [18]:
# Hitung scale_pos_weight untuk LightGBM
neg = (y == 0).sum()
pos = (y == 1).sum()
scale_pos_weight = neg / pos
print(f"scale_pos_weight: {scale_pos_weight:.2f}")
# Ini yang handle imbalance, jauh lebih efisien dari SMOTE (SMOTE = RAM killer)

scale_pos_weight: 27.58


### Training

In [19]:
# LightGBM params
params = {
  'objective': 'binary',
  'metric': 'auc',
  'boosting_type': 'gbdt',
  'num_leaves': 256,
  'max_depth': -1,
  'learning_rate': 0.05,
  'n_estimators': 1000,
  'subsample': 0.8,
  'subsample_freq': 1,
  'colsample_bytree': 0.8,
  'min_child_samples': 100,
  'scale_pos_weight': scale_pos_weight,
  'reg_alpha': 0.1,
  'reg_lambda': 0.1,
  'n_jobs': -1,
  'random_state': 42,
  'verbose': -1,
  # Memory optimization
  'max_bin': 63,  # default 255
}

# 5-Fold Stratified CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
feature_importance = pd.DataFrame()

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
  print(f"\n=== Fold {fold+1}/5 ===")

  X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
  y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

  model = lgb.LGBMClassifier(**params)
  model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[
      lgb.early_stopping(stopping_rounds=50, verbose=False),
      lgb.log_evaluation(period=100)
    ]
  )

  oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
  test_preds += model.predict_proba(X_test)[:, 1] / 5

  fold_auc = roc_auc_score(y_val, oof_preds[val_idx])
  print(f"Fold {fold+1} AUC: {fold_auc:.4f}")

  # Feature importance
  fi = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_,
    'fold': fold+1
  })
  feature_importance = pd.concat([feature_importance, fi], ignore_index=True)

  del X_train, X_val, y_train, y_val
  gc.collect()

overall_auc = roc_auc_score(y, oof_preds)
print(f"\n=== Overall OOF AUC: {overall_auc:.4f} ===")


=== Fold 1/5 ===
[100]	valid_0's auc: 0.955659
[200]	valid_0's auc: 0.963737
[300]	valid_0's auc: 0.966867
[400]	valid_0's auc: 0.968376
[500]	valid_0's auc: 0.969192
[600]	valid_0's auc: 0.969688
[700]	valid_0's auc: 0.970007
[800]	valid_0's auc: 0.9704
[900]	valid_0's auc: 0.970653
Fold 1 AUC: 0.9707

=== Fold 2/5 ===
[100]	valid_0's auc: 0.956944
[200]	valid_0's auc: 0.965123
[300]	valid_0's auc: 0.968152
[400]	valid_0's auc: 0.969796
[500]	valid_0's auc: 0.970796
[600]	valid_0's auc: 0.971297
[700]	valid_0's auc: 0.971619
[800]	valid_0's auc: 0.972077
Fold 2 AUC: 0.9721

=== Fold 3/5 ===
[100]	valid_0's auc: 0.953971
[200]	valid_0's auc: 0.962498
[300]	valid_0's auc: 0.965755
[400]	valid_0's auc: 0.967458
[500]	valid_0's auc: 0.96907
[600]	valid_0's auc: 0.969518
[700]	valid_0's auc: 0.970226
[800]	valid_0's auc: 0.97048
Fold 3 AUC: 0.9706

=== Fold 4/5 ===
[100]	valid_0's auc: 0.955233
[200]	valid_0's auc: 0.963985
[300]	valid_0's auc: 0.968107
[400]	valid_0's auc: 0.970183
[500]

### Evaluation Metrics

# Metrik yang digunakan:
  1. ROC-AUC  -> metrik utama untuk fraud detection (threshold-agnostic)
  2. PR-AUC   -> lebih informatif untuk kelas imbalanced
  </br> (fokus pada precision-recall trade-off di kelas minoritas)
  3. Classification Report -> precision, recall, F1 per kelas
  4. Confusion Matrix -> lihat false positive & false negative

Threshold default 0.5 dipakai untuk classification report.

In [20]:
from sklearn.metrics import (
  classification_report, confusion_matrix,
  precision_recall_curve, average_precision_score
)
import matplotlib.pyplot as plt

# Convert prob ke binary prediction (threshold 0.5)
oof_binary = (oof_preds >= 0.5).astype(int)

print("=== Classification Report ===")
print(classification_report(y, oof_binary))

print(f"\nROC-AUC Score: {overall_auc:.4f}")
print(f"Average Precision Score: {average_precision_score(y, oof_preds):.4f}")

# Confusion Matrix
cm = confusion_matrix(y, oof_binary)
print(f"\nConfusion Matrix:\n{cm}")

# Feature Importance Plot
fi_mean = feature_importance.groupby('feature')['importance'].mean().sort_values(ascending=False)
top_features = fi_mean.head(30)

plt.figure(figsize=(10, 10))
top_features.plot(kind='barh')
plt.title('Top 30 Feature Importances')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=80)  # savefig hemat RAM vs plt.show()
plt.close()

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.99      1.00      0.99    569877
           1       0.86      0.77      0.81     20663

    accuracy                           0.99    590540
   macro avg       0.92      0.88      0.90    590540
weighted avg       0.99      0.99      0.99    590540


ROC-AUC Score: 0.9717
Average Precision Score: 0.8597

Confusion Matrix:
[[567204   2673]
 [  4737  15926]]


### Hyperparameter Tuning

Tuning dilakukan pada 30% subset data untuk hemat waktu & RAM.
Parameter yang di-tune:
  - num_leaves  : mengontrol kompleksitas model
  - learning_rate: mengontrol kecepatan belajar

Trade-off:
  - num_leaves tinggi + lr kecil = lebih akurat, lebih lambat
  - num_leaves rendah + lr besar = lebih cepat, risiko underfit

3-fold CV dipakai (bukan 5) supaya lebih cepat di subset data.

In [21]:
# Manual grid search pada subset data

from sklearn.model_selection import cross_val_score

X_sample = X.sample(frac=0.3, random_state=42)  # pakai 30% data untuk tuning
y_sample = y[X_sample.index]

tuning_results = []
for num_leaves in [64, 128, 256]:
  for lr in [0.05, 0.1]:
    p = params.copy()
    p['num_leaves'] = num_leaves
    p['learning_rate'] = lr
    p['n_estimators'] = 300  # lebih kecil untuk tuning

    model_tune = lgb.LGBMClassifier(**p)
    scores = cross_val_score(
      model_tune, X_sample, y_sample,
      cv=3, scoring='roc_auc', n_jobs=-1
    )
    tuning_results.append({
      'num_leaves': num_leaves,
      'learning_rate': lr,
      'mean_auc': scores.mean(),
      'std_auc': scores.std()
    })
    print(f"num_leaves={num_leaves}, lr={lr} -> AUC: {scores.mean():.4f} ± {scores.std():.4f}")

tuning_df = pd.DataFrame(tuning_results).sort_values('mean_auc', ascending=False)
print("\nBest params:")
print(tuning_df.iloc[0])

num_leaves=64, lr=0.05 -> AUC: 0.9335 ± 0.0016
num_leaves=64, lr=0.1 -> AUC: 0.9335 ± 0.0021
num_leaves=128, lr=0.05 -> AUC: 0.9360 ± 0.0018
num_leaves=128, lr=0.1 -> AUC: 0.9353 ± 0.0019
num_leaves=256, lr=0.05 -> AUC: 0.9376 ± 0.0021
num_leaves=256, lr=0.1 -> AUC: 0.9340 ± 0.0020

Best params:
num_leaves       256.000000
learning_rate      0.050000
mean_auc           0.937649
std_auc            0.002122
Name: 4, dtype: float64


### Submission

Output: CSV dengan 2 kolom:
  - TransactionID : ID transaksi dari test set
  - isFraud       : probabilitas fraud (bukan label biner)

test_preds adalah rata-rata prediksi probabilitas dari 5 fold

In [22]:
submission = pd.DataFrame({
  'TransactionID': test_ids,
  'isFraud': test_preds
})
submission.to_csv('submission.csv', index=False)
print(submission.head())
print(f"Submission shape: {submission.shape}")

   TransactionID   isFraud
0        3663549  0.000535
1        3663550  0.000417
2        3663551  0.000794
3        3663552  0.000744
4        3663553  0.000416
Submission shape: (506691, 2)
